# Session 1c: Attention to TinyGPT

**Course:** Language Models: ML Basics to Modern AI (BTU Cottbus, M.Sc. AI seminar)
**Session:** 1 of 4, notebook 1c of 3
**Lecture reference:** Lecture on self-attention, multi-head attention, and the transformer block.

## Learning objectives

By the end of this notebook you should be able to:

- Derive scaled dot-product attention from query, key, and value tensors and explain the role of the `sqrt(d_k)` divisor.
- Implement multi-head attention as `h` parallel attention computations on projected subspaces of `d_model`.
- Assemble a transformer block (pre-LN residual stream, multi-head attention sublayer, position-wise feed-forward sublayer) and stack `n_layers` of them into a `TinyGPT` decoder.
- Run a forward pass and a greedy (or temperature-sampled) `generate` on a randomly initialised model, and recognise that the resulting text is gibberish until training closes the loop.
- Save your `TinyGPT` class to `tiny_gpt.py` for reuse in Session 2's pre-training notebook.

The notebook accompanies the lecture on self-attention. It does not re-derive the theory. It makes the theory concrete by building the decoder from primitives.


## §1 Primer: from dot products to transformer blocks

Self-attention is the operation that lets a transformer mix information across positions in a sequence. The lecture covered the diagram and the equations. The notebook builds them in code. The primer below collects the pieces you need before writing the implementation, in the order they show up in the build.

### Scaled dot-product attention

Imagine each position in the sequence as a small lookup operation. The token at position $i$ asks a question and wants to retrieve information from the other positions to answer it. Three vectors per position make this concrete: a query $q_i$ that encodes what is being asked, a key $k_j$ at each candidate position $j$ that advertises what that position can offer, and a value $v_j$ that holds the actual information to be retrieved. The query is matched against every key, the matches turn into mixing weights, and the values are averaged under those weights to produce the answer.

Each of $q_i$, $k_j$, $v_j$ has dimension $d_k$ and is produced by a learned linear projection of the input. Stacking row-wise gives $Q, K, V$ of shape $(T, d_k)$. The match between $q_i$ and $k_j$ is the dot product $q_i \cdot k_j$: large when the two vectors point in similar directions, small when they do not. Compute every pair at once and you get a $(T, T)$ matrix of raw scores. The full operation is

$$\text{Attention}(Q, K, V) = \text{softmax}\!\left(\frac{Q K^\top}{\sqrt{d_k}}\right) V.$$

Read it in three steps. $Q K^\top$ is the $(T, T)$ score matrix. The row-wise softmax turns each row into a probability distribution over positions: row $i$ tells us how to mix when computing the output at $i$. The product with $V$ takes that mixing distribution and applies it to the value vectors, producing a $(T, d_k)$ output: one mixed value per position.

### Why divide by $\sqrt{d_k}$

The naive attention formula sums $d_k$ products to compute each score. As $d_k$ grows, those sums get larger just from accumulation: a wider head produces louder scores even when nothing about the input has changed. The next step, the softmax, responds to absolute score gaps, not relative ones, so louder scores produce sharper distributions whether they should or not. By the time $d_k$ reaches 64, a typical raw score sits around $\pm 8$, and softmax with inputs in that range behaves almost like a hard argmax: the largest score swallows nearly all the probability mass, every other position gets a vanishing gradient, and the model loses the signal that would tell it which positions actually matter.

Dividing by $\sqrt{d_k}$ undoes the accumulation. Formally, if the entries of $q$ and $k$ are independent with zero mean and unit variance, then $q \cdot k = \sum_{i=1}^{d_k} q_i k_i$ has variance $d_k$, so its standard deviation grows as $\sqrt{d_k}$. The divisor cancels that growth and keeps the scores at unit scale regardless of head width. Softmax then operates in its responsive regime: small changes in the input produce small changes in the output, gradients propagate to many positions, and the model can actually learn which keys match which queries instead of locking onto a winner from initialisation.

### What softmax actually does to the scores

Attention needs a way to pick. Without softmax the model would average every value with equal weight and could never focus on a particular position. With softmax, the exponential map turns small score differences into decisive choices: a score gap of one unit puts about 73% of the mass on the higher-scoring position, a gap of two units 88%, a gap of four units 98%. This sharpness is the lookup mechanism. It is what lets a pronoun token at position $i$ pull most of its mixed output from the antecedent noun at some earlier position rather than averaging across the whole context.

The same sharpness is also the failure mode when scores grow too large. Softmax cannot tell whether two scores differ by 2 or by 20; both saturate to roughly the same one-hot distribution. But only the smaller gap produces gradients large enough for the loser to recover. If raw dot products produce gaps of 20 units, the model commits to its initial guess before it has any reason to and never updates. Sharpness is a feature when it follows from evidence, a bug when it is baked into the geometry. The $\sqrt{d_k}$ rescaling is what keeps it the first.

### Multi-head attention

A single attention head produces one mixing scheme per position: one way of deciding "given my query at position $i$, which other positions should I copy from?" One scheme is not enough for natural language. A pronoun-resolution head wants to point at the nearest plausible antecedent. A coreference head wants to point at the topical noun many words back. An in-context-learning head wants to copy from an earlier (input, label) pair in the prompt. These are different functions of the same input, and one mixing scheme cannot do all three at once because the softmax of a single attention head sums to one: probability spent on the antecedent is probability not available for anything else.

Multi-head attention runs $h$ such schemes in parallel. Each head has its own $d_k = d_{\text{model}} / h$ subspace, runs its own softmax, and the outputs are concatenated and projected back. The projections sit in a single $(d_{\text{model}}, 3 \, d_{\text{model}})$ linear layer (the `qkv` projection in the code); the heads are split out by reshaping; scaled dot-product attention runs once on the batched-with-heads tensor; an output projection $(d_{\text{model}}, d_{\text{model}})$ mixes the concatenated head outputs. The total parameter budget matches a single attention head at full $d_{\text{model}}$ width. Splitting it across $h$ subspaces buys $h$ independent mixing schemes for the same compute and memory. The §3 widget shows this in action: most heads of a trained BERT specialise for one specific relationship type, and you can see it in the patterns.

### Causal masking

A decoder predicts the next token from the prefix. Position $i$ must not see positions $j > i$, or the model learns a function it cannot use at inference: at training time the answer is in the context, at inference time it is not, and the two distributions stop matching. The model would learn to peek at the next token and the loss would look great until the first inference call.

The standard fix is a $(T, T)$ lower-triangular mask that zeroes out the forbidden entries before softmax. In code the mask is applied by filling the upper triangle of the scores with $-\infty$, so softmax sends those entries to exactly zero. The same mask applies to every head and every layer; the upper-triangular structure is the property that defines a causal (decoder) transformer.

### Residual connections and where to put LayerNorm

Each sublayer (attention, then feed-forward) is wrapped in a residual connection: $x \leftarrow x + f(x)$. The residual gives the block one job: compute a correction to the running representation, not a full replacement. The running representation lives on the residual stream, the additive sum $x + f_1(x) + f_2(\dots) + \dots$ that threads through the entire network from the input embedding to the final logits. Gradients flow back along this stream essentially unimpeded by depth, which is why you can stack dozens of transformer blocks without the network becoming untrainable. Residuals also let each block be small in effect: it adds, it does not overwrite.

LayerNorm placement matters because two conventions exist. The original 2017 transformer used post-LN, where the norm comes after the residual add: $x \leftarrow \text{LN}(x + f(x))$. This means every block re-normalises the residual stream itself, and gradients through many layers pass through many LayerNorms in series. With deep stacks, post-LN destabilises early in training and needs a long learning-rate warmup to recover. Modern decoders use pre-LN instead: $x \leftarrow x + f(\text{LN}(x))$. Only the input to the sublayer is normalised; the residual stream itself is never touched. The gradient highway along the residual stream is clean of LN, and only the sublayer-input gradient sees the norm. The result is stable training without warmup tricks, which is what makes pre-LN the default for everything modern. `TinyGPT` is pre-LN.

### The position-wise feed-forward sublayer

Attention mixes information across positions. Up to that point the per-position transformation is purely linear, since the QKV and output projections are linear maps. The feed-forward sublayer is where each position gets a non-linear transformation of its own representation: linear up to a wider hidden dimension (typically $4 \, d_{\text{model}}$), GELU non-linearity, linear back down to $d_{\text{model}}$. The widening is doing real work. A two-layer MLP at constant width would compose into a single linear map and add nothing the attention projections do not already provide; the wider middle dimension is the room in which the non-linearity actually changes the function class. Empirically, about two-thirds of a transformer's parameter budget lives in these feed-forward layers, not in attention.

### Tying input and output embeddings

Two operations involve the same vocabulary: the input embedding maps a token ID to a $d_{\text{model}}$ vector, and the output projection (the LM head) maps a $d_{\text{model}}$ vector back to a distribution over vocabulary. Both weight matrices have shape $(V, d_{\text{model}})$. Sharing them is the line `self.head.weight = self.embedder.token.weight`.

Sharing has two effects. Mechanically, it halves the parameter count of the largest single layer in a small model and (empirically) trains more stably. Semantically, it enforces a consistency that makes the model easier to interpret: the vector that represents the token "cat" on the way in is the same vector the model points at when predicting "cat" on the way out. Untied embeddings would let those two diverge during training, which is wasteful since they encode the same object from two angles. After tying, gradients from the cross-entropy loss flow into the same matrix from both ends.

You now have everything for the build. The §2 setup cell imports torch and pins the seed. The §3 widget shows what attention looks like after training in BERT. The §4 warm-ups exercise the two primitives you just read about (the scaled softmax and the causal mask). The §5 deep build assembles `scaled_dot_product_attention`, `MultiHeadAttention`, `TransformerBlock`, and `TinyGPT` in that order, then runs `generate` on a randomly initialised model so you can see what untrained attention produces. The two §5.5 primers cover the two engineering tricks (KV caching and flash attention) that turn this textbook implementation into a production-grade kernel. Session 2 then trains the same model on a small corpus.


In [ ]:
"""§2 Setup: imports, random seed, device selection."""

import math
import warnings

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn.functional as F
from torch import nn

warnings.filterwarnings("ignore")

SEED = 0
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"PyTorch {torch.__version__} on {DEVICE}")


## §3 Guided exploration: what trained attention looks like

The widget below loads `bert-base-uncased` and renders the twelve attention heads of one layer as a 3 by 4 grid of heatmaps. Each heatmap shows, for the chosen sentence, how much every token at every position attends to every other position. This is the pattern you are about to build the mechanism for in §5.

Three parts. Part 1 takes any sentence and shows all twelve heads at a layer of your choice. Part 2 walks through six linguistic phenomena (pronoun resolution, gender bias, word-sense disambiguation, long-range dependencies, subject-verb agreement) with pre-selected sentences and layer hints. Part 3 holds the sentence fixed and sweeps the layer slider, so the progression from local diagonal patterns in early layers to long-range semantic patterns in late layers is visible at a glance.

The first call downloads roughly 440 MB of BERT weights. If the download fails (no network, a sandboxed environment), the cell prints a clearly marked notice and falls back to a synthetic random attention tensor of plausible shape. The widgets still run in that mode but the patterns are noise; only the structural layout (12 heads, 12 layers, a square per layer) is honest. The §4 warm-ups and the §5 deep build do not depend on BERT in any way.


In [ ]:
"""§3 BERT attention heatmap widget.

Three parts (your sentence, guided examples, layer progression). Falls back to a
synthetic random attention tensor if the BERT download fails, so the cell runs
end-to-end regardless of network availability.
"""

import base64
import io as _io

import ipywidgets as widgets
from IPython.display import HTML, display

_BERT_OK = True
_BERT_NOTICE = ""

try:
    from transformers import BertModel, BertTokenizer

    tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")
    bert_model = BertModel.from_pretrained("bert-base-uncased", output_attentions=True)
    bert_model.eval()
except Exception as _err:
    _BERT_OK = False
    _BERT_NOTICE = (
        f"[fallback] BERT load failed ({type(_err).__name__}: {_err}). "
        f"The widget will display a synthetic stand-in attention map of shape "
        f"(12 layers, 12 heads, T, T). The patterns are random and not interpretable; "
        f"the structural layout is preserved so the UI remains usable."
    )
    print(_BERT_NOTICE)
    tokenizer = None
    bert_model = None


def _simple_tokenize(sentence: str) -> list:
    """Whitespace tokenise plus BERT-style sentence markers. Used by the fallback path."""
    words = sentence.strip().split()
    return ["[CLS]"] + words + ["[SEP]"]


def get_attention(sentence: str):
    """Return (attn, tokens). attn has shape (n_layers=12, n_heads=12, T, T)."""
    if _BERT_OK:
        enc = tokenizer(sentence, return_tensors="pt")
        toks = tokenizer.convert_ids_to_tokens(enc["input_ids"][0])
        with torch.no_grad():
            out = bert_model(**enc)
        attn = torch.stack(out.attentions).squeeze(1).numpy()
        return attn, toks
    # Synthetic fallback: random softmaxed scores at the right shape.
    toks = _simple_tokenize(sentence)
    T = len(toks)
    rng = np.random.default_rng(abs(hash(sentence)) % (2**32))
    raw = rng.standard_normal((12, 12, T, T)).astype(np.float32)
    # Row-wise softmax so each row is a valid distribution.
    raw = raw - raw.max(axis=-1, keepdims=True)
    exp = np.exp(raw)
    attn = exp / exp.sum(axis=-1, keepdims=True)
    return attn, toks


def _fig_to_html(fig):
    buf = _io.BytesIO()
    fig.savefig(
        buf, format="png", bbox_inches="tight",
        facecolor=fig.get_facecolor(), dpi=100,
    )
    buf.seek(0)
    b64 = base64.b64encode(buf.read()).decode()
    plt.close(fig)
    return HTML(
        '<img src="data:image/png;base64,' + b64
        + '" style="max-width:100%;border-radius:8px">'
    )


def _clean(t: str) -> str:
    return (
        t.replace("##", "")
         .replace("[CLS]", ">")
         .replace("[SEP]", "|")
    )


def plot_heads(attn, tokens, layer_idx, hl_head=None):
    """Render a 3x4 grid of attention heatmaps for one layer."""
    attn_l = attn[layer_idx]
    seq = min(len(tokens), 14)
    labels = [_clean(t) for t in tokens[:seq]]

    fig, axes = plt.subplots(3, 4, figsize=(14, 9))
    fig.patch.set_facecolor("#0f172a")
    fig.suptitle(
        f"Layer {layer_idx + 1} of 12, all 12 attention heads",
        fontsize=13, color="white", fontweight="bold", y=1.01,
    )

    for h in range(12):
        ax = axes[h // 4][h % 4]
        ax.set_facecolor("#1e293b")
        A = attn_l[h][:seq, :seq]
        ihl = (hl_head is not None and h == hl_head)
        ax.imshow(
            A, cmap="RdPu" if ihl else "Blues",
            vmin=0, vmax=max(float(A.max()), 1e-6), aspect="auto",
        )
        fs = 6.5 if seq <= 12 else 5.5
        ax.set_xticks(range(seq))
        ax.set_yticks(range(seq))
        ax.set_xticklabels(labels, rotation=90, fontsize=fs, color="#cbd5e1")
        ax.set_yticklabels(labels, fontsize=fs, color="#cbd5e1")
        col = "#f472b6" if ihl else "#94a3b8"
        ax.set_title(
            f"Head {h + 1}", fontsize=9, color=col,
            fontweight="bold" if ihl else "normal",
        )
        for s in ax.spines.values():
            s.set_edgecolor("#f472b6" if ihl else "#334155")
            s.set_linewidth(2.5 if ihl else 0.5)

    plt.tight_layout()
    return fig


# --- Part 1: your sentence --------------------------------------------------------
_inp1 = widgets.Text(
    value="The trophy did not fit in the suitcase because it was too big.",
    layout=widgets.Layout(width="96%"),
    style={"description_width": "0px"},
)
_lyr1 = widgets.IntSlider(
    value=8, min=0, max=11, description="Layer:",
    layout=widgets.Layout(width="55%"),
    style={"description_width": "50px"},
)
_btn1 = widgets.Button(
    description="  Visualise", button_style="primary",
    layout=widgets.Layout(width="155px"), icon="eye",
)
_info1 = widgets.HTML(value="")
_handle1 = None


def _run1(b=None):
    if _handle1 is None:
        return
    s = _inp1.value.strip()
    if not s:
        return
    _info1.value = '<span style="color:#64748b;font-size:12px">computing...</span>'
    try:
        attn, toks = get_attention(s)
        L = _lyr1.value
        mode = "BERT" if _BERT_OK else "synthetic fallback"
        _info1.value = (
            f'<span style="color:#4ade80;font-size:12px">'
            f'{len(toks)} tokens, layer {L + 1} of 12 ({mode})</span>'
        )
        _handle1.update(_fig_to_html(plot_heads(attn, toks, L)))
    except Exception as e:
        _info1.value = f'<span style="color:#f87171">Error: {e}</span>'


_btn1.on_click(_run1)
_lyr1.observe(lambda c: _run1(), names="value")


# --- Part 2: guided linguistic examples -------------------------------------------
_EX = [
    dict(
        label="1. Pronoun resolution: trophy vs. suitcase",
        sentence="The trophy did not fit in the suitcase because it was too big.",
        layer=9,
        hint=(
            "Find a head where <b>'it'</b> attends to <b>'trophy'</b> rather than "
            "<b>'suitcase'</b>. Start at layers 9 to 12."
        ),
    ),
    dict(
        label="2. Gender bias: nurse vs. doctor",
        sentence="The nurse told the doctor that she was exhausted.",
        layer=9,
        hint=(
            "Watch which noun <b>'she'</b> attends to. This reveals the bias the "
            "model has absorbed from its training data."
        ),
    ),
    dict(
        label="3. Word sense: financial 'bank'",
        sentence="I went to the bank to deposit my savings.",
        layer=10,
        hint=(
            "In late layers, <b>'bank'</b> tends to attend to <b>'deposit'</b> and "
            "<b>'savings'</b>. These context words anchor its financial sense."
        ),
    ),
    dict(
        label="4. Word sense: river 'bank' (contrast with example 3)",
        sentence="I sat by the bank of the river all afternoon.",
        layer=10,
        hint=(
            "Compare to example 3. Same surface form, different context, different "
            "attention pattern."
        ),
    ),
    dict(
        label="5. Long-range dependency: cat / dog / big",
        sentence="The cat that the dog chased was very big.",
        layer=9,
        hint=(
            "Find a head where <b>'big'</b> attends to <b>'cat'</b>, skipping the "
            "relative clause. Sequential models struggle with this. Attention does not."
        ),
    ),
    dict(
        label="6. Subject-verb agreement: keys vs. cabinet",
        sentence="The keys to the cabinet are on the table.",
        layer=6,
        hint=(
            "Find a head where <b>'are'</b> attends to <b>'keys'</b> (plural) rather "
            "than <b>'cabinet'</b> (singular). Grammatical number tracking across an "
            "intervening noun phrase."
        ),
    ),
]

_labels2 = [e["label"] for e in _EX]
_dd2 = widgets.Dropdown(
    options=_labels2, value=_labels2[0],
    layout=widgets.Layout(width="96%"),
    style={"description_width": "0px"},
)
_lyr2 = widgets.IntSlider(
    value=9, min=0, max=11, description="Layer:",
    layout=widgets.Layout(width="55%"),
    style={"description_width": "50px"},
)
_hint2 = widgets.HTML(value="")
_handle2 = None


def _draw2(change=None):
    if _handle2 is None:
        return
    idx = _labels2.index(_dd2.value)
    ex = _EX[idx]
    _hint2.value = (
        '<div style="background:#1e3a5f;border:1px solid #3b82f6;'
        'border-radius:8px;padding:10px 14px;margin:6px 0;'
        'font-size:13px;color:#e2e8f0">'
        '<span style="color:#93c5fd"><b>Sentence:</b></span> '
        f'<i style="color:#f1f5f9">{ex["sentence"]}</i><br><br>'
        f'{ex["hint"]}</div>'
    )
    try:
        attn, toks = get_attention(ex["sentence"])
        _handle2.update(_fig_to_html(plot_heads(attn, toks, _lyr2.value)))
    except Exception as e:
        _handle2.update(HTML(f'<p style="color:#f87171">Error: {e}</p>'))


def _on_dd2(change=None):
    idx = _labels2.index(_dd2.value)
    _lyr2.value = _EX[idx]["layer"]
    _draw2()


_dd2.observe(_on_dd2, names="value")
_lyr2.observe(_draw2, names="value")


# --- Part 3: layer progression on a fixed sentence -------------------------------
_SENT3 = "The trophy did not fit in the suitcase because it was too big."

_NOTES3 = [
    "Layer 1. Very local. Punctuation and immediately adjacent tokens dominate.",
    "Layer 2. Still local. Diagonal patterns (each token to its neighbours).",
    "Layer 3. Positional patterns emerge: tokens attending at fixed relative offsets.",
    "Layer 4. Early syntactic structure begins to form.",
    "Layer 5. Mix of syntactic and early semantic signals.",
    "Layer 6. Subject-verb agreement heads often appear here.",
    "Layer 7. Pronouns begin attending to candidate antecedents.",
    "Layer 8. Coreference patterns strengthen.",
    "Layer 9. Long-range semantic connections clearly visible.",
    "Layer 10. Representations are contextualised and semantically rich.",
    "Layer 11. Abstract, high-level patterns. Local details recede.",
    "Layer 12. Final layer. Richest semantic signal, feeds the prediction head.",
]

_lyr3 = widgets.IntSlider(
    value=0, min=0, max=11, description="Layer:",
    continuous_update=True,
    layout=widgets.Layout(width="65%"),
    style={"description_width": "50px"},
)
_note3 = widgets.HTML(value="")
_handle3 = None

_attn3, _tok3 = get_attention(_SENT3)


def _run3(change=None):
    if _handle3 is None:
        return
    L = _lyr3.value
    _note3.value = (
        '<div style="background:#172554;border:1px solid #3b82f6;'
        'border-radius:6px;padding:8px 12px;font-size:13px;'
        f'color:#bfdbfe;margin:4px 0">{_NOTES3[L]}</div>'
    )
    _handle3.update(_fig_to_html(plot_heads(_attn3, _tok3, L)))


_lyr3.observe(_run3, names="value")


# --- Render all three parts ------------------------------------------------------
# Each part's interactive plot is anchored at a stable display_id and updated in
# place via a DisplayHandle. This avoids ipywidgets.Output's accumulation behaviour
# under static renderers (VS Code in particular) where the cleared outputs would
# still be saved in the widget state and re-render on each load.

# Part 1: render header, controls, and initial plot.
display(widgets.HTML('<h4 style="margin-top:4px">Part 1: BERT attention on a sentence you choose</h4>'))
display(widgets.VBox([
    _inp1,
    widgets.HBox([_lyr1, widgets.HTML("&nbsp;&nbsp;"), _btn1]),
    _info1,
], layout=widgets.Layout(padding="4px")))

_attn1_init, _tok1_init = get_attention(_inp1.value.strip())
_info1.value = (
    f'<span style="color:#4ade80;font-size:12px">'
    f'{len(_tok1_init)} tokens, layer {_lyr1.value + 1} of 12 '
    f'({"BERT" if _BERT_OK else "synthetic fallback"})</span>'
)
_handle1 = display(
    _fig_to_html(plot_heads(_attn1_init, _tok1_init, _lyr1.value)),
    display_id="bert-attn-part1",
)


# Part 2: render header, controls, and initial plot at the first example.
display(widgets.HTML("<h4>Part 2: guided linguistic examples</h4>"))
display(widgets.VBox([_dd2, _lyr2, _hint2],
                     layout=widgets.Layout(padding="4px")))

_ex_init = _EX[0]
_hint2.value = (
    '<div style="background:#1e3a5f;border:1px solid #3b82f6;'
    'border-radius:8px;padding:10px 14px;margin:6px 0;'
    'font-size:13px;color:#e2e8f0">'
    '<span style="color:#93c5fd"><b>Sentence:</b></span> '
    f'<i style="color:#f1f5f9">{_ex_init["sentence"]}</i><br><br>'
    f'{_ex_init["hint"]}</div>'
)
_attn2_init, _tok2_init = get_attention(_ex_init["sentence"])
_handle2 = display(
    _fig_to_html(plot_heads(_attn2_init, _tok2_init, _lyr2.value)),
    display_id="bert-attn-part2",
)


# Part 3: render header, slider, and initial plot at layer 1.
display(widgets.HTML("<h4>Part 3: layer progression on a fixed sentence</h4>"))
display(widgets.HTML(
    f'<p style="font-size:14px;color:#e2e8f0;margin:4px 0 8px">'
    f'<b>Sentence:</b> <i>"{_SENT3}"</i></p>'
    '<p style="font-size:13px;color:#94a3b8;margin-bottom:4px">'
    "Drag the slider from left (layer 1) to right (layer 12) and watch how "
    "the attention patterns evolve.</p>"
))
display(widgets.VBox([_lyr3, _note3],
                     layout=widgets.Layout(padding="4px")))

_note3.value = (
    '<div style="background:#172554;border:1px solid #3b82f6;'
    'border-radius:6px;padding:8px 12px;font-size:13px;'
    f'color:#bfdbfe;margin:4px 0">{_NOTES3[_lyr3.value]}</div>'
)
_handle3 = display(
    _fig_to_html(plot_heads(_attn3, _tok3, _lyr3.value)),
    display_id="bert-attn-part3",
)


## §4 Warm-ups

Two short exercises before the deep build. The first computes attention weights from given Q, K, V tensors and confirms the row sums equal one. The second constructs the causal mask used by every decoder.


In [ ]:
"""§4 Warm-up 1 (exercise): attention weights from given Q, K, V.

Given the tensors `q`, `k`, `v` below, compute the unnormalised attention scores
`scores = q @ k^T / sqrt(d_k)` and the softmax weights along the last dimension.
After implementing, the printed row sums should each be 1.0.
"""

import math

import torch
import torch.nn.functional as F

torch.manual_seed(0)
B, H, T, Dk = 1, 1, 4, 8
q = torch.randn(B, H, T, Dk)
k = torch.randn(B, H, T, Dk)
v = torch.randn(B, H, T, Dk)

# TODO: compute `scores` as scaled dot product of q and k along the last dim
# TODO: compute `weights` as softmax of scores along the last dim
# scores = ...
# weights = ...

# After implementing, uncomment:
# print("scores shape:", scores.shape)
# print("weights row sums (should be 1.0):", weights.sum(dim=-1))

try:
    scores  # noqa: F821
    weights  # noqa: F821
except NameError:
    print("scores / weights not computed yet.")


In [ ]:
"""§4 Warm-up 2 (exercise): causal mask for a length-T sequence.

Implement `causal_mask(T)` so that it returns a (1, 1, T, T) tensor with 1s on
the lower triangle (including the diagonal) and 0s above. This mask is what
prevents a decoder from attending to future positions.

Hint: torch.tril.
"""

import torch


def causal_mask(T: int) -> torch.Tensor:
    """Return a (1, 1, T, T) lower-triangular mask of 0s and 1s."""
    # TODO: build the (T, T) lower-triangular mask, then reshape to (1, 1, T, T)
    raise NotImplementedError


try:
    print(causal_mask(4))
except NotImplementedError:
    print("causal_mask not implemented yet.")


## §5 Deep build: scaled attention to TinyGPT

The next five cells assemble the decoder bottom-up. Subtask 1 is the scaled dot-product attention function from the primer. Subtask 2 wraps it in `MultiHeadAttention` with the qkv-fused projection. Subtask 3 puts attention and a feed-forward sublayer inside a pre-LN residual block. Subtask 4 stacks `n_layers` blocks, adds the input embedder and the tied output head, and gives the module a `generate` method. Subtask 5 instantiates a small model and runs `generate` on a prompt.

The final output of subtask 5 will be gibberish. The model has never seen a single training example; the weights are still at their PyTorch initialisation. That gibberish is the point: it is what an architecturally correct but untrained decoder looks like. Closing the gap between this output and coherent English is the job of Session 2.


### Before you implement: think through the mask

The TODO below asks you to apply a mask "before softmax" so that masked positions end up with **zero weight after softmax**. Two natural-looking ideas do *not* work:

- **Zeroing the weights after softmax** breaks normalisation: each row no longer sums to 1, so the output is no longer a weighted average of the values.
- **Setting the scores to 0 before softmax** gives those positions weight $\exp(0) = 1$ &mdash; the opposite of what you want.

Recall the definition:

$$\text{softmax}(x_i) = \frac{\exp(x_i)}{\sum_j \exp(x_j)}.$$

For a position to receive **exactly zero** weight, its numerator $\exp(x_i)$ must be zero. What value of $x_i$ forces $\exp(x_i)$ to zero? That is the value you should substitute into the score matrix at masked positions, *before* softmax.

In [ ]:
"""§5 Subtask 1 (exercise): scaled dot-product attention.

Implement the textbook attention computation:

    Attention(Q, K, V) = softmax(Q @ K^T / sqrt(d_k)) @ V

with optional masking. If `mask` is provided, entries where the mask is 0 must
contribute zero weight after softmax (set the corresponding scores to -inf
before softmax).

Shapes:
    q, k, v: (..., T, d_k)
    mask:    broadcastable to (..., T_q, T_k). 0 means masked.
"""

import math

import torch
import torch.nn.functional as F


def scaled_dot_product_attention(
    q: torch.Tensor,
    k: torch.Tensor,
    v: torch.Tensor,
    mask: torch.Tensor | None = None,
) -> torch.Tensor:
    """Standard scaled dot-product attention."""
    # TODO: implement Attention(Q, K, V) = softmax(QK^T / sqrt(d_k)) V with optional
    # masking. Positions where mask == 0 must contribute zero weight after softmax.
    #
    # Hint: softmax(x_i) = exp(x_i) / sum_j exp(x_j). What value of x_i forces the
    # numerator to zero? Apply this value to scores BEFORE softmax, not after.
    raise NotImplementedError


try:
    _q = torch.randn(2, 4, 8, 16)
    _k = torch.randn(2, 4, 8, 16)
    _v = torch.randn(2, 4, 8, 16)
    _out = scaled_dot_product_attention(_q, _k, _v)
    print("Output shape:", tuple(_out.shape))
except NotImplementedError:
    print("scaled_dot_product_attention not implemented yet.")

In [ ]:
"""§5 Subtask 2 (exercise): multi-head attention.

Implement MultiHeadAttention as `n_heads` independent attention computations on
projected subspaces of `d_model`. Use a single fused linear layer for Q, K, V
(output dim `3 * d_model`) and reshape into heads. After scaled_dot_product_attention
runs on the batched-with-heads tensor, combine the heads back to `d_model` and
apply the output projection.

Required attributes (so the rest of the notebook works):
    self.n_heads, self.d_head, self.qkv (Linear), self.out (Linear)
"""

from torch import nn


class MultiHeadAttention(nn.Module):
    def __init__(self, d_model: int, n_heads: int) -> None:
        super().__init__()
        # TODO: define the four attributes the rest of the notebook expects:
        #   n_heads, d_head, qkv (a Linear that produces Q, K, V at once), out (Linear).
        # Reject configurations where d_model is not divisible by n_heads.
        raise NotImplementedError

    def forward(
        self, x: torch.Tensor, mask: torch.Tensor | None = None
    ) -> torch.Tensor:
        # TODO: produce Q, K, V from x via self.qkv and split them across n_heads.
        # Run scaled_dot_product_attention with the supplied mask, then project the
        # concatenated heads back to d_model with self.out.
        #
        # Three steps before attention can run:
        #   1. Split the qkv output (shape B, T, 3*d_model) into three tensors of
        #      shape (B, T, d_model). torch.chunk or torch.split along the last dim.
        #   2. Reshape each to (B, T, n_heads, d_head).
        #   3. Permute so the head dim sits next to the batch dim: (B, n_heads, T, d_head).
        # After scaled_dot_product_attention, reverse steps 3 and 2 to get back to
        # (B, T, d_model), then project with self.out.
        raise NotImplementedError


try:
    _mha = MultiHeadAttention(d_model=32, n_heads=4)
    _x = torch.randn(2, 8, 32)
    print("MultiHeadAttention output shape:", tuple(_mha(_x).shape))
except NotImplementedError:
    print("MultiHeadAttention not implemented yet.")

In [ ]:
"""§5 Subtask 3 (exercise): the pre-LN transformer block.

Build a TransformerBlock with two sublayers in the pre-LN configuration:

    x = x + attn(LN(x), mask=mask)
    x = x + ffn(LN(x))

The feed-forward sublayer widens from d_model to ffn_mult * d_model with a GELU
nonlinearity and projects back down. A dropout layer at the end of the FFN is
standard.
"""


class TransformerBlock(nn.Module):
    def __init__(
        self,
        d_model: int,
        n_heads: int,
        ffn_mult: int = 4,
        dropout: float = 0.0,
    ) -> None:
        super().__init__()
        # TODO: define the two LayerNorms (ln1, ln2), the MultiHeadAttention sublayer
        # (attn), and the FFN sublayer (ffn). The FFN widens d_model by ffn_mult with a
        # GELU nonlinearity and projects back to d_model, followed by Dropout(dropout).
        raise NotImplementedError

    def forward(
        self, x: torch.Tensor, mask: torch.Tensor | None = None
    ) -> torch.Tensor:
        # TODO: implement the two pre-LN residual sublayers (LN, then sublayer, then
        # add to x). The structure is in this cell's docstring.
        raise NotImplementedError


try:
    _block = TransformerBlock(d_model=32, n_heads=4)
    _x = torch.randn(2, 8, 32)
    print("TransformerBlock output shape:", tuple(_block(_x).shape))
except NotImplementedError:
    print("TransformerBlock not implemented yet.")


In [ ]:
"""§5 Subtask 4 (exercise): the TinyGPT decoder with tied weights and generate().

Build the full decoder: input embedder, n_layers transformer blocks, final
LayerNorm, output projection. Tie the output projection weights to the input
embedding weights: `self.head.weight = self.embedder.token.weight`.

In `forward`, build the (1, 1, T, T) lower-triangular causal mask on the input
device and pass it to every block.

In `generate`, for `max_new_tokens` iterations:
    - crop the context to the last self.max_seq_len tokens,
    - run forward to get logits, take the last-position slice,
    - divide by temperature, optionally apply top-k,
    - softmax + multinomial sample, concatenate to idx.

The helper `_sinusoidal_positional_encoding` and the `TokenEmbedder` class are
provided inline so the cell is self-contained.
"""


def _sinusoidal_positional_encoding(max_len: int, d_model: int) -> torch.Tensor:
    position = torch.arange(max_len).unsqueeze(1).float()
    div_term = torch.exp(
        torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model)
    )
    pe = torch.zeros(max_len, d_model)
    pe[:, 0::2] = torch.sin(position * div_term)
    pe[:, 1::2] = torch.cos(position * div_term)
    return pe


class TokenEmbedder(nn.Module):
    """Token embedding plus sinusoidal positional encoding."""

    def __init__(self, vocab_size: int, d_model: int, max_seq_len: int = 1024) -> None:
        super().__init__()
        self.token = nn.Embedding(vocab_size, d_model)
        self.register_buffer(
            "pos", _sinusoidal_positional_encoding(max_seq_len, d_model)
        )

    def forward(self, ids: torch.Tensor) -> torch.Tensor:
        _, t = ids.shape
        return self.token(ids) + self.pos[:t]


class TinyGPT(nn.Module):
    def __init__(
        self,
        vocab_size: int,
        d_model: int = 128,
        n_heads: int = 4,
        n_layers: int = 4,
        max_seq_len: int = 256,
        dropout: float = 0.0,
    ) -> None:
        super().__init__()
        # TODO: assemble the model. Five attributes the rest of the notebook expects:
        #   - max_seq_len  (the cap on generated context length)
        #   - embedder     (a TokenEmbedder)
        #   - blocks       (an nn.ModuleList of n_layers TransformerBlocks)
        #   - ln_f         (a final LayerNorm before the output projection)
        #   - head         (the linear projection to vocab logits, no bias)
        # The output head's weight should be tied to the input embedding weight.
        # See the lecture or the §1 primer for the rationale.
        raise NotImplementedError

    def forward(self, idx: torch.Tensor) -> torch.Tensor:
        # TODO: embed the input ids, build a causal mask, run the stack of blocks,
        # apply the final LayerNorm, and project to vocab logits of shape (B, T, V).
        raise NotImplementedError

    @torch.no_grad()
    def generate(
        self,
        idx: torch.Tensor,
        max_new_tokens: int,
        temperature: float = 1.0,
        top_k: int | None = None,
    ) -> torch.Tensor:
        # TODO: implement autoregressive decoding. At each step:
        #   - crop the context to the last max_seq_len tokens,
        #   - run forward, take the last-position logits,
        #   - divide by temperature and (if top_k) keep only the top-k logits,
        #   - sample one new token from the softmax of the resulting logits,
        #   - append it to idx.
        # Return the extended idx tensor.
        #
        # Hint on top-k: "keep only the top-k logits" means setting all non-top-k
        # logits to -inf so they collapse to zero after softmax (same trick as the
        # attention mask). Use torch.topk to find the threshold value, then mask.
        # Hint on sampling: torch.multinomial(probs, num_samples=1) draws one token
        # from a categorical distribution defined by `probs`.
        raise NotImplementedError


try:
    _m = TinyGPT(vocab_size=100, d_model=32, n_heads=4, n_layers=2, max_seq_len=16)
    print("TinyGPT forward shape:", tuple(_m(torch.randint(0, 100, (2, 8))).shape))
except NotImplementedError:
    print("TinyGPT not implemented yet.")

In [ ]:
"""§5 Subtask 5 (exercise): instantiate, generate, observe gibberish.

This cell only runs once TinyGPT (Subtask 4) is implemented. Before that, it
prints a short message and exits cleanly.
"""

try:
    TinyGPT  # noqa: F821
except NameError:
    print("Implement TinyGPT in Subtask 4 first.")
else:
    try:
        torch.manual_seed(0)
        from transformers import AutoTokenizer

        tok = AutoTokenizer.from_pretrained("gpt2")
        model = TinyGPT(
            vocab_size=tok.vocab_size,
            d_model=64,
            n_heads=4,
            n_layers=2,
            max_seq_len=64,
        )
        prompt = tok("Once upon a time", return_tensors="pt")["input_ids"]
        out = model.generate(prompt, max_new_tokens=20, temperature=1.0)
        print(tok.decode(out[0]))
        # Expected: gibberish. The model has not been trained.
    except NotImplementedError:
        print("TinyGPT or one of its dependencies still has a NotImplementedError body.")


## §5.5 KV cache: paying for inference once instead of every step

Autoregressive decoding feeds the model its own output one token at a time. At step $t$ the input is positions $0$ through $t-1$ and the model produces a distribution over the token at position $t$. The naive implementation re-runs the full forward pass on the entire prefix, so the per-step cost grows as $O(n^2)$ and total decoding cost as $O(N^3)$.

The fix is to notice that the keys and values at positions $0$ through $t-1$ depend only on the input at those positions and the model weights. Neither has changed since step $t-1$. Cache them. At step $t$ the model computes the K and V for the new position and runs attention against the cache. Per-token cost drops to $O(n \cdot d_{\text{model}})$, and the total cost of generating $N$ tokens drops to $O(N^2)$. The query is computed only for the new position, since past queries are never reused.

Typical layout: one cache tensor per layer, shape `(batch, n_heads, max_seq_len, d_head)`, allocated once at the maximum sequence length and filled row by row as decoding proceeds.

```
position:    0   1   2   3   4   5   6   7   ...
step 0:      K0
step 1:      K0  K1
step 2:      K0  K1  K2
step 3:      K0  K1  K2  K3
step 4:      K0  K1  K2  K3  K4
```

Memory becomes the bottleneck. A 7B model at sequence length 4096 in fp16 needs roughly 2 GB of cache per request, separate from the weights. The KV cache, not the parameters, is what makes long-context inference expensive in production.


## §5.5 Flash attention: tiling, fusion, and SRAM

The textbook attention computation materialises the full $(T, T)$ score matrix in GPU global memory, runs softmax along each row, then multiplies the result by $V$. For sequence length $T = 8{,}192$ in fp16 the intermediate matrix alone is 128 MB per head per layer. With many heads and many layers, memory dominates the cost long before the arithmetic does, and the kernel spends most of its time moving data between SRAM and HBM rather than computing.

Flash attention rearranges the computation to never materialise the score matrix in global memory. The sequence is tiled into blocks of, say, 128 queries and 128 keys at a time; the corresponding score sub-block, its softmax, and the value multiply all happen inside on-chip SRAM in a single fused pass. A running tally of the row-wise maximum and partial sum lets the algorithm produce the same result as the naive implementation by stitching tiles together. The mathematical output is bit-equivalent (modulo floating-point reassociation). The memory cost is linear in $T$ rather than quadratic.

Flash attention powers both training and the prefill step of inference (the one-shot forward pass over the input prompt before decoding begins). The decode step is dominated by the KV cache pattern from the previous section, where each step processes a single new token and the score matrix is a thin slice. Production systems combine both: flash attention for prefill, KV cache for decode.

In PyTorch the production-grade kernel is `torch.nn.functional.scaled_dot_product_attention`, which dispatches to a flash-attention or memory-efficient backend on the GPU automatically. Use it whenever you write attention code that matters; the textbook implementation in §5 is for learning.


## §6 Recap and next step

You have implemented the decoder side of the transformer end-to-end: scaled dot-product attention with optional masking, multi-head attention with the fused qkv projection, a pre-LN transformer block, and the full `TinyGPT` module with weight tying and a `generate` method. The forward pass produces the correct shapes and `generate` produces text. The text is currently gibberish because the weights are randomly initialised.

The canonical copy of this module lives at `student/_reference/tiny_gpt.py` in your seminar package. Session 2's pre-training notebook imports it from there, fits the weights on a small corpus, and the same `generate` call you ran above starts producing coherent text.


In [ ]:
"""§6 Save module step (informational).

Writing files from a notebook is fragile (the working directory depends on
how the notebook was launched). In your seminar package the canonical copy
of the module lives at student/_reference/tiny_gpt.py.
"""

print("The canonical copy of your TinyGPT module lives at")
print("    student/_reference/tiny_gpt.py")
print("It is identical to what you just built. Session 2 imports it from there.")
